In [0]:
df_silver = spark.read.table("he_prod_incen_analytics_dbw_01.silver.fact_pharmacy")
df_dim_members = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_members")
df_dim_providers = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_providers")
df_dim_date = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_date")

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, coalesce, lit, col

df_gold = df_silver.withColumn("pharmacy_sk", monotonically_increasing_id() + 1)

df_gold = df_gold.join(df_dim_members.select(col("member_sk").alias("gold_member_sk"), "member_id"), on="member_id", how="left").withColumn("gold_member_sk", coalesce(col("gold_member_sk"), lit(-1)))
df_gold = df_gold.join(df_dim_providers.select(col("provider_sk").alias("gold_provider_sk"), "provider_id"), on="provider_id", how="left").withColumn("gold_provider_sk", coalesce(col("gold_provider_sk"), lit(-1)))
df_gold = df_gold.join(df_dim_date.select(col("date_sk").alias("gold_date_sk"), col("date").alias("fill_date")), on="fill_date", how="left").withColumn("gold_date_sk", coalesce(col("gold_date_sk"), lit(-1)))

df_gold = df_gold.select(
    "pharmacy_sk", "pharmacy_claim_id",
    col("gold_member_sk").alias("member_sk"), col("gold_provider_sk").alias("provider_sk"), 
    col("gold_date_sk").alias("fill_date_sk"),
    "ndc_code", "drug_name", "generic_flag", "quantity_dispensed", "gross_cost", "insurance_paid"
)

display(df_gold.limit(5))

In [0]:
df_gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("he_prod_incen_analytics_dbw_01.gold.fact_pharmacy")
print("✅ Successfully wrote Fact_Pharmacy to Gold layer.")